# Goal

Выбор оптимального edge_loss_coef, bce_loss_coef ставим 1.

# set_hyperparameters

In [1]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.system.random_seed = random.randint(1, 100)
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    HP.system.use_amp = True
    
    HP.model.parent = None
    HP.model.d_model = 256
    HP.model.layers_count = 3
    HP.model.heads_count = 4
    HP.model.actions_count = 6
    HP.model.ob_shape = (1, 178, 152)
    HP.model.ob_grid_shape = (16, 16)
    HP.model.sequence_length = 4
    HP.model.attention_backend = 'EFFICIENT_ATTENTION'
    
    HP.dataset.train = [
        'train_dataset:100',
        'train_dataset:101',
        'train_dataset:102',
        'train_dataset:103',
        'train_dataset:104',
        'train_dataset:105',
        'train_dataset:106',
        'train_dataset:107',
        'train_dataset:108',
        'train_dataset:109',
    ]
    HP.dataset.test = 'test_dataset:2'
    
    HP.train.epochs_count = 100
    HP.train.batch_size = 128
    HP.train.optimizer = 'AdamW'
    HP.train.max_grad_norm = 1.0
    HP.train.learn_rate = 'const(0.0001)'
    
    HP.train.bce_loss_coef = f'const(1.0)'
    edge_loss_coef = optuna_trial.suggest_float('train.edge_loss_coef', 0.5, 5)
    HP.train.edge_loss_coef = f'const({edge_loss_coef})'
    
    return HP
# @launchit.stop

# Results
<TBD>

Как ни странно, но лучше всего оказался запуск 17:
> Trial 17 (18b_world_model_01:42) finished with value: 0.9459326367937128 and parameters: {'train.edge_loss_coef': 0.5572186552703187}. Best is trial 17 with value: 0.9459326367937128
Launch "18b_world_model_01:42" completed

<img src="./img/ssim.png">

Т.е. bce_loss_coef=1, edge_loss_coef=0.557. Я думал, что нужен более высокий edge_coef. Но это, если смотреть на ssim. Что-то я подозреваю, что сама по себе эта метрика не позволит оценить, насколько хорошо модель предсказывает будущее. Для неё нахождение игрока/противника плюс-минус несколько пикселей кажется будет незначимо, тогда как для логики игры - это критично. Т.е. вес объектов на картинке, он не одинаковый - какие-то объекты неважны, какие-то супер важны. А у нас тут просто смотрится то, насколько картинки структурно похожи.

Проблематика видна, если смотреть на картинка в тензорборде по 42-му запуску. Всякие льдины и прочее отрисовано неплохо, но вот игрок, гуси и рыба - просто какое-то пушистое говно.

**Выводы**
1) само собой напрашивается добавить в датасет вектор значимых переменных (размер иглу, позиция игрока, сколько жизней осталось, позиция врагов и т.д.). По факту - вырезку из RAM.
2) но это уже какой-то читинг, т.к. в реальной жизни доступ к этой информации закрыт - модель и должна её найти в UNSUPERVISED режиме. А тут будет supervised режим.
3) ещё меня мучает тема, что генерация выходной картинки несколько "туповата". Кажется, что один пайплайн из ConvTranspose не способоен нарисовать такую сложную сцену и нужен ансамбль.
4) короче, сначал сделаем 3.